# FIFA World Cup — Data Exploration & EDA

This notebook covers **Steps 1 & 2** of the project:

1. **Explore the data** — inspect every CSV, explain its contents, identify the
   most useful tables, propose joins, and surface data-quality issues.
2. **Exploratory analysis** — historical results, goals, draws, host effect,
   confederations, tournament stages, and Elo-based team strength.

Source dataset: [jfjelstul/worldcup](https://github.com/jfjelstul/worldcup).

> **Scope decision:** we model the **men's** World Cup only. `team_id`s are
> shared between the men's and women's datasets, so mixing them would corrupt
> team-strength estimates. All modeling uses men's tournaments (1930–2022).

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# make the src package importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config, data, elo  # noqa: E402

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
FIG = config.FIGURES_DIR
print("Project root:", ROOT)

Project root: /Users/lucasdidone/Documents/fifa_world_cup_test


## 1. Dataset overview — what each file contains

The dataset ships one CSV per entity. Below is a size summary, followed by a
description of each file and how useful it is for **match-outcome prediction**.

In [2]:
summary = data.data_summary()
summary

,file,rows,cols,n_missing
0,award_winners.csv,200,12,0
1,awards.csv,8,5,0
2,bookings.csv,3178,26,0
3,confederations.csv,6,5,0
4,fixtures_2026.csv,72,5,0
5,goals.csv,3637,27,0
6,group_standings.csv,626,19,0
7,groups.csv,159,7,0
8,host_countries.csv,31,7,0
9,manager_appearances.csv,2538,17,0


### File-by-file description & usefulness

| File | Grain | Contents | Usefulness for prediction |
|---|---|---|---|
| **`matches.csv`** | 1 row / match | teams, date, stage, scores, result flags, ET/penalties | **CORE** — the label source and basis of all features |
| **`team_appearances.csv`** | 1 row / team / match | long form of matches (goals_for/against, result per team) | **CORE (alt. view)** — convenient for per-team rolling stats |
| **`tournaments.csv`** | 1 row / tournament | year, host country, winner, #teams, stages played | **High** — host indicator, temporal splits, tournament format |
| **`teams.csv`** | 1 row / team | name, code, **confederation**, federation, region | **High** — confederation features, name↔id mapping |
| `tournament_stages.csv` | 1 row / stage | stage names & ordering per tournament | Medium — stage encoding |
| `tournament_standings.csv` | 1 row / team / tournament | final position | Medium — could build "best finish" experience |
| `group_standings.csv`, `groups.csv` | group tables | group results | Low — derivable from matches |
| `goals.csv` | 1 row / goal | scorer, minute, own-goal/penalty | Low for outcome; useful for goal-timing analysis |
| `squads.csv`, `players.csv`, `player_appearances.csv` | player level | rosters, line-ups | Low here (no ratings); needed for player-level models |
| `managers*.csv`, `referees*.csv` | staff | managers / referees & appointments | Low — possible niche features |
| `bookings.csv`, `substitutions.csv`, `penalty_kicks.csv` | in-match events | cards, subs, shoot-out kicks | Low — post-match / in-play, leakage risk |
| `award_winners.csv`, `awards.csv` | awards | Golden Ball etc. | None for pre-match prediction |
| `confederations.csv` | 1 row / confederation | confederation metadata | Low — lookup |
| `host_countries.csv`, `stadiums.csv`, `qualified_teams.csv` | misc | hosting, venues, qualification | Low/Medium — host & venue context |

**Bottom line:** `matches` + `tournaments` + `teams` are sufficient to build a
sound, leakage-free outcome model. `team_appearances` is a convenient
ready-made long view. Player/event tables are out of scope for a results-only
model but are the natural place to extend it later.

### Relationships & how to join

```
tournaments (tournament_id) 1───* matches (tournament_id)
teams       (team_id)       1───* matches (home_team_id / away_team_id)
matches     (match_id)      1───2 team_appearances (match_id, team_id)
```

* `matches.home_team_id` / `away_team_id` → `teams.team_id` (confederation join).
* `matches.tournament_id` → `tournaments.tournament_id` (host, year, format).
* `team_appearances` is `matches` exploded to two rows per match (one per team),
  which makes per-team rolling features trivial.

**Neutral-venue note:** in `matches`, `home_team`/`away_team` only encode
*listing order* (World Cup venues are neutral). We must not treat "home" as a
real advantage; see the bias check below.

In [3]:
# Core cleaned table (men's only, chronologically sorted, replays handled)
matches = data.load_men_matches()
print(matches.shape)
matches[["tournament_name", "match_date", "stage_name", "home_team_name",
         "away_team_name", "home_team_score", "away_team_score",
         "result", "penalty_shootout"]].head()

(960, 24)


,tournament_name,match_date,stage_name,home_team_name,away_team_name,home_team_score,away_team_score,result,penalty_shootout
0,1930 FIFA Men's World Cup,1930-07-13,group stage,France,Mexico,4,1,home team win,0
1,1930 FIFA Men's World Cup,1930-07-13,group stage,United States,Belgium,3,0,home team win,0
2,1930 FIFA Men's World Cup,1930-07-14,group stage,Yugoslavia,Brazil,2,1,home team win,0
3,1930 FIFA Men's World Cup,1930-07-14,group stage,Romania,Peru,3,1,home team win,0
4,1930 FIFA Men's World Cup,1930-07-15,group stage,Argentina,France,1,0,home team win,0


### Data-quality checks

We check: missing values, duplicate matches, inconsistent team names, the
handful of replayed matches, and how penalty shoot-outs are encoded (this last
point directly affects how we define the target).

In [4]:
raw_matches = pd.read_csv(config.RAW_DIR / "matches.csv")

print("Missing values in matches.csv:", int(raw_matches.isna().sum().sum()))
print("Duplicate match_ids:", int(raw_matches["match_id"].duplicated().sum()))

# Replayed matches (annulled draws that were replayed)
print("\nReplayed / replay flags:", raw_matches["replayed"].sum(),
      raw_matches["replay"].sum())

# How are penalty shoot-outs encoded? -> result records the SHOOT-OUT WINNER,
# but home/away scores hold the regulation+ET draw. We therefore define the
# target from goals (penalty matches = draws in open play).
ps = raw_matches[raw_matches["penalty_shootout"] == 1]
print("\nPenalty shoot-out matches:", len(ps))
print(ps[["match_name", "home_team_score", "away_team_score",
          "score_penalties", "result", "draw"]].head())

# Name consistency: confederation coverage for teams that actually played
teams = data.load_teams()
print("\nTeams missing a confederation:",
      int(teams["confederation_code"].isna().sum()))

Missing values in matches.csv: 0
Duplicate match_ids: 0

Replayed / replay flags: 4 4

Penalty shoot-out matches: 43
                         match_name  home_team_score  away_team_score  \
357          West Germany vs France                3                3   
404                Brazil vs France                1                1   
405          West Germany vs Mexico                0                0   
407                Spain vs Belgium                1                1   
452  Republic of Ireland vs Romania                0                0   

    score_penalties         result  draw  
357             5–4  home team win     0  
404             3–4  away team win     0  
405             4–1  home team win     0  
407             4–5  away team win     0  
452             5–4  home team win     0  

Teams missing a confederation: 0


**Data-quality findings**

* **No missing values** in `matches.csv`; goal columns are clean integers.
* **No duplicate `match_id`s**. A few historical matches were *replayed*
  (annulled draws); `load_men_matches()` keeps only the decisive leg.
* **Penalty shoot-outs:** `result` records the shoot-out *winner*, but the
  goal columns hold the *regulation+ET draw*. We define the target from goals,
  so a shoot-out counts as a **draw** (the correct 1X2 / open-play result).
* **Team names** are internally consistent (`team_id`-keyed). The only mismatch
  risk is between our *2026 fixture spelling* and the dataset (e.g. dataset uses
  *"South Korea"*, *"Ivory Coast"*); handled by an alias table in `src.predict`.
* **Defunct teams** (West Germany, USSR, Yugoslavia, Czechoslovakia) are distinct
  `team_id`s; their successors do not inherit ratings — a known limitation.

## 2. Exploratory data analysis

### 2.1 Outcome distribution & the listing-order bias

If we naively use the raw `home`/`away` labels, "home" teams win far more often.
But venues are neutral — this is an artefact of stronger/seeded teams being
*listed first*. This is the single most important leakage trap in this dataset.

In [5]:
raw_result = matches["result"].value_counts()
goal_result = np.where(matches["home_team_score"] > matches["away_team_score"],
                       "listed-team win",
                       np.where(matches["home_team_score"] == matches["away_team_score"],
                                "draw", "other-team win"))
goal_counts = pd.Series(goal_result).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw_result.plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Raw 'result' (listing order)\n-> spurious home bias")
axes[0].tick_params(axis="x", rotation=20)
goal_counts.plot(kind="bar", ax=axes[1], color="#C44E52")
axes[1].set_title("Goal-based outcome by listing order")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(FIG / "outcome_distribution.png", dpi=110); plt.show()

print("Listed-team win rate:", round((matches['home_team_score'] >
      matches['away_team_score']).mean(), 3))
print("Draw rate:", round((matches['home_team_score'] ==
      matches['away_team_score']).mean(), 3))

Listed-team win rate: 0.546
Draw rate: 0.219


/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_6868/1469611372.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "outcome_distribution.png", dpi=110); plt.show()


The ~55% / 22% / 23% split for the *listed* team confirms the bias. Our
modeling dataset fixes this by **randomly assigning** `team_a`/`team_b` per
match, which yields a balanced ≈39% / 22% / 39% target — exactly what we want
for a neutral-venue model.

### 2.2 Goals: how scoring evolved over time

In [6]:
matches["total_goals"] = matches["home_team_score"] + matches["away_team_score"]
matches["is_draw"] = (matches["home_team_score"] == matches["away_team_score"])
by_year = matches.groupby("year").agg(
    avg_goals=("total_goals", "mean"),
    draw_rate=("is_draw", "mean"),
    n=("match_id", "count"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(by_year["year"], by_year["avg_goals"], "o-", color="#4C72B0",
         label="avg goals/match")
ax1.set_ylabel("avg goals / match", color="#4C72B0")
ax1.axhline(matches["total_goals"].mean(), ls="--", c="grey", alpha=.6)
ax2 = ax1.twinx()
ax2.plot(by_year["year"], by_year["draw_rate"], "s-", color="#C44E52",
         label="draw rate")
ax2.set_ylabel("draw rate", color="#C44E52")
ax1.set_title("Goals per match & draw rate by tournament")
plt.tight_layout(); plt.savefig(FIG / "goals_draws_over_time.png", dpi=110); plt.show()

# Distribution of goals per team-match
fig, ax = plt.subplots(figsize=(8, 4))
gf = pd.concat([matches["home_team_score"], matches["away_team_score"]])
sns.histplot(gf, bins=range(0, 12), discrete=True, ax=ax, color="#55A868")
ax.set_title(f"Goals scored per team per match (mean={gf.mean():.2f})")
ax.set_xlabel("goals"); plt.tight_layout()
plt.savefig(FIG / "goals_per_team_hist.png", dpi=110); plt.show()

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_6868/3896309527.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "goals_draws_over_time.png", dpi=110); plt.show()
/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_6868/3896309527.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(FIG / "goals_per_team_hist.png", dpi=110); plt.show()


Scoring was very high in the early tournaments (1950s peak > 4 goals/match) and
settled around **2.5–2.8 goals/match** in the modern era. Per-team goals are
well-described by a low-mean count distribution — motivating the **Poisson**
goal model. Draws hover around **20–25%**.

### 2.3 Host effect

Does hosting help? We compare host teams' win rate to the overall baseline.

In [7]:
# Build a long team-match view with a host flag
hosts = data.host_country_map()
ta = pd.read_csv(config.RAW_DIR / "team_appearances.csv")
ta = ta[ta["tournament_name"].str.contains("Men's")].copy()
ta["year"] = pd.to_datetime(ta["match_date"]).dt.year
ta["is_host"] = ta.apply(lambda r: r["team_name"] in hosts.get(int(r["year"]), set()),
                         axis=1)
ta["points"] = ta["result"].map({"win": 3, "draw": 1, "lose": 0})

host_stats = ta.groupby("is_host").agg(
    win_rate=("win", "mean"),
    avg_points=("points", "mean"),
    avg_goals_for=("goals_for", "mean"),
    n=("win", "count"),
)
display(host_stats)

# host best-finishes
t = data.load_tournaments()
print("Host won the tournament:", int(t["host_won"].sum()), "of", len(t),
      "tournaments")

,win_rate,avg_points,avg_goals_for,n
is_host,,,,
False,0.391400,1.364388,1.381477,1814
True,0.657895,2.087719,1.877193,114


Host won the tournament: 7 of 30 tournaments


Hosts clearly outperform: higher win rate, points and goals than the average
team, and several hosts have won the tournament outright. This justifies a
**host indicator** feature (note: 2026 has three hosts — USA, Mexico, Canada).

### 2.4 Confederation strength

In [8]:
conf = data.load_teams()[["team_id", "confederation_code"]]
ta2 = ta.merge(conf, on="team_id", how="left")
conf_stats = (ta2.groupby("confederation_code")
              .agg(win_rate=("win", "mean"), avg_points=("points", "mean"),
                   matches=("win", "count"))
              .sort_values("win_rate", ascending=False))

fig, ax = plt.subplots(figsize=(8, 4))
conf_stats["win_rate"].plot(kind="bar", ax=ax, color="#8172B3")
ax.set_title("Win rate by confederation (men's WC, all matches)")
ax.set_ylabel("win rate"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig(FIG / "confederation_winrate.png", dpi=110); plt.show()
conf_stats

/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_6868/3839543777.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "confederation_winrate.png", dpi=110); plt.show()


,win_rate,avg_points,matches
confederation_code,,,
CONMEBOL,0.515789,1.702632,380
UEFA,0.451524,1.542013,1083
CAF,0.234568,0.944444,162
CONCACAF,0.227273,0.870130,154
AFC,0.188811,0.741259,143
OFC,0.000000,0.500000,6


UEFA and CONMEBOL dominate (every champion has come from one of them). This
motivates a `same_confederation` context feature and confirms confederation
membership carries signal — though it is largely *correlated with* team
strength, which Elo already captures.

### 2.5 Tournament stage & Elo-based team strength

In [9]:
# Goals & draw rate by stage type (group vs knockout)
stage_stats = (matches.assign(stage=np.where(matches["knockout_stage"] == 1,
                                             "knockout", "group"))
               .groupby("stage")
               .agg(avg_goals=("total_goals", "mean"),
                    draw_rate=("is_draw", "mean"), n=("match_id", "count")))
display(stage_stats)

# Elo: final ratings of the strongest teams + a few trajectories
ratings = elo.final_ratings(matches)
teams_lkp = data.load_teams().set_index("team_id")["team_name"].to_dict()
top = (pd.Series({teams_lkp.get(k, k): v for k, v in ratings.items()})
       .sort_values(ascending=False).head(15))
fig, ax = plt.subplots(figsize=(9, 4))
top.iloc[::-1].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Top 15 teams by final Elo (end of 2022)")
ax.set_xlabel("Elo"); ax.set_xlim(1450, top.max() + 30)
plt.tight_layout(); plt.savefig(FIG / "top_elo.png", dpi=110); plt.show()

,avg_goals,draw_rate,n
stage,,,
group,2.692201,0.243733,718
knockout,3.202479,0.144628,242


/var/folders/5m/0wrpl7jd3sj7_dzwy_3t8m0r0000gn/T/ipykernel_6868/3040676701.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "top_elo.png", dpi=110); plt.show()


### Key EDA insights (summary)

1. **Listing-order bias is real and dangerous.** The raw "home" team wins ~55%
   of matches purely because stronger teams are listed first. We neutralise this
   by randomising `team_a`/`team_b`, giving a balanced ~39/22/39 target.
2. **Draws (~20–25%) are the hard, irreducible class.** Models rarely pick draw
   as the *argmax*, so we judge them by **probabilistic** metrics (log loss /
   Brier), not accuracy.
3. **Scoring is low-count & Poisson-like** (~2.6 goals/match in the modern era),
   justifying a Poisson goal model for score prediction.
4. **Host advantage is substantial** → host indicator feature (3 hosts in 2026).
5. **UEFA & CONMEBOL dominate** — every champion comes from these two.
6. **Knockout matches are lower-scoring with more draws** than group games →
   a `knockout_stage` feature.
7. **Elo summarises strength well** and is fully derivable from results, making
   it our strongest single, leakage-free feature.

➡️ These insights directly shape the feature set and the modeling/validation
choices in `02_modeling_and_predictions.ipynb`.